<h1>Introduction to dplyr</h1>

<small>Source: <a href="https://github.com/tidyverse/dplyr/blob/pkgdown-v1.1.4/vignettes/dplyr.Rmd"><code>vignettes/dplyr.Rmd</code></a></small>
<div><code>dplyr.Rmd</code></div>


<p>When working with data you must:</p>

<ul>
<li><p>Figure out what you want to do.</p></li>
<li><p>Describe those tasks in the form of a computer program.</p></li>
<li><p>Execute the program.</p></li>
</ul>

<p>The dplyr package makes these steps fast and easy:</p>

<ul>
<li><p>By constraining your options, it helps you think about your data manipulation challenges.</p></li>
<li><p>It provides simple “verbs”, functions that correspond to the most common data manipulation tasks, to help you translate your thoughts into
code.</p></li>
<li><p>It uses efficient backends, so you spend less time waiting for the computer.</p></li>
</ul>

<p>This document introduces you to dplyr’s basic set of tools, and shows you how to apply them to data frames. dplyr also supports databases via
the dbplyr package, once you’ve installed, read <code>vignette("dbplyr")</code> to learn more.</p>

# <h2 id="data-starwars">Data: starwars</h2>

<p>To explore the basic data manipulation verbs of dplyr, we’ll use the dataset <code>starwars</code>. This dataset contains 87 characters and
comes from the <a href="https://swapi.dev">Star Wars API</a>, and is documented in <code><a href="https://dplyr.tidyverse.org/reference/starwars.html">?starwars</a></code></p>

In [4]:
library(dplyr)

In [5]:
dim(starwars)

[1] 87 14

In [6]:
starwars

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


<p>Note that <code>starwars</code> is a tibble, a modern reimagining of the data frame. It’s particularly useful for large datasets because it
only prints the first few rows. You can learn more about tibbles at <a href="https://tibble.tidyverse.org">https://tibble.tidyverse.org</a>; in particular you can convert data frames to tibbles with <code><a href="https://tibble.tidyverse.org/reference/as_tibble.html">as_tibble()</a></code>.</p>


# <h2 id="single-table-verbs">Single table verbs</h2>

<p>dplyr aims to provide a function for each basic verb of data manipulation. These verbs can be organised into three categories based on the component of the dataset that they work with:</p>

<ul>
<li>Rows:
<ul>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> chooses rows based on column values.</li>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice()</a></code> chooses rows based on location.</li>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange()</a></code> changes the order of the rows.</li>
</ul>
</li>
<li>Columns:
<ul>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> changes whether or not a column is
included.</li>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/rename.html">rename()</a></code> changes the name of columns.</li>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> changes the values of columns and creates new
columns.</li>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/relocate.html">relocate()</a></code> changes the order of the columns.</li>
</ul>
</li>
<li>Groups of rows:
<ul>
<li>
<code><a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise()</a></code> collapses a group into a single row.</li>
</ul>
</li>
</ul>


## <h3 id="the-pipe">The pipe</h3>

<p>All of the dplyr functions take a data frame (or tibble) as the first argument. Rather than forcing the user to either save intermediate
objects or nest functions, dplyr provides the <code>%&gt;%</code> operator from magrittr. <code>x %&gt;% f(y)</code> turns into <code>f(x, y)</code> so the result from one step is then “piped” into the next step. You can use the pipe to rewrite multiple operations that you can read left-to-right, top-to-bottom (reading the pipe operator as “then”).</p>


## <h3 id="filter-rows-with-filter">Filter rows with <code>filter()</code></h3>

<p><code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> allows you to select a subset of rows in a data frame. Like all single verbs, the first argument is the tibble (or data frame). The second and subsequent arguments refer to variables within that data frame, selecting rows where the expression is <code>TRUE</code>.</p>
    
<p>For example, we can select all character with light skin color and brown eyes with:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/filter.html">filter</a>(<skin_color == "light", <eye_color == "brown")
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 7 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Leia O…    150    49 brown      light      brown             19 fema…
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Biggs …    183    84 black      light      brown             24 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Padmé …    185    45 brown      light      brown             46 fema…
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Cordé      157    <span style="color: #BB0000;">NA brown      light      brown             <span style="color: #BB0000;">NA <span style="color: #BB0000;">NA   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 3 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre></div>

<p>This is roughly equivalent to this base R code:</p>

<pre><code>starwars[starwars$skin_color == "light" &amp; starwars$eye_color == "brown", ]</code></pre>

## <h3 id="arrange-rows-with-arrange">Arrange rows with <code>arrange()</code></h3>

<p><code><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange()</a></code> works similarly to <code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> except that instead of filtering or selecting rows, it reorders them. It takes a data frame, and a set of column names (or more complicated expressions) to order by. If you provide more than one column name, each additional column will be used to break ties in the values of preceding columns:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/arrange.html">arrange</a>(<height, <mass)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Yoda        66    17 white      green      brown            896 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Ratts …     79    15 none       grey, blue unknown           <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Wicket…     88    20 brown      brown      brown              8 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Dud Bo…     94    45 none       blue, grey yellow            <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

<p>Use <code><a href="https://dplyr.tidyverse.org/reference/desc.html">desc()</a></code> to order a column in descending order:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/arrange.html">arrange</a>(<a href="../reference/desc.html">desc</a>(<height))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Yarael…    264    <span style="color: #BB0000;">NA none       white      yellow            <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Tarfful    234   136 brown      brown      blue              <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Lama Su    229    88 none       grey       black             <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Chewba…    228   112 brown      unknown    blue             200 male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

## <h3 id="choose-rows-using-their-position-with-slice">Choose rows using their position with <code>slice()</code></h3>

<p><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice()</a></code> lets you index rows by their (integer) locations. It allows you to select, remove, and duplicate rows.</p>

<p>We can get characters from row numbers 5 through 10.</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/slice.html">slice</a>(<span class="fl">5:<span class="fl">10)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 6 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Leia O…    150    49 brown      light      brown             19 fema…
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Owen L…    178   120 brown, gr… light      blue              52 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Beru W…    165    75 brown      light      blue              47 fema…
<span class="co">#&gt; <span style="color: #BCBCBC;">4 R5-D4       97    32 <span style="color: #BB0000;">NA         white, red red               <span style="color: #BB0000;">NA none 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 2 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

<p>It is accompanied by a number of helpers for common use cases:</p>

<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_head()</a></code> and <code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_tail()</a></code> select the first or last rows.</li>
</ul>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/slice.html">slice_head</a>(n = <span class="fl">3)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 3 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke S…    172    77 blond      fair       blue              19 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow           112 none 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red               33 none 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_sample()</a></code> randomly selects rows. Use the option prop to choose a certain proportion of the cases.</li>
</ul>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/slice.html">slice_sample</a>(n = <span class="fl">5)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 5 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Ayla S…    178  55   none       blue       hazel             48 fema…
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Bossk      190 113   none       green      red               53 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 San Hi…    191  <span style="color: #BB0000;">NA   none       grey       gold              <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Lumina…    170  56.2 black      yellow     blue              58 fema…
<span class="co">#&gt; <span style="color: #949494;"># ℹ 1 more row
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;
<starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/slice.html">slice_sample</a>(prop = <span class="fl">0.1)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 8 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Qui-Go…    193    89 brown      fair       blue              92 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Jango …    183    79 black      tan        brown             66 male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Jocast…    167    <span style="color: #BB0000;">NA white      fair       blue              <span style="color: #BB0000;">NA fema…
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Zam We…    168    55 blonde     fair, gre… yellow            <span style="color: #BB0000;">NA fema…
<span class="co">#&gt; <span style="color: #949494;"># ℹ 4 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

<p>Use <code>replace = TRUE</code> to perform a bootstrap sample. If needed, you can weight the sample with the <code>weight</code> argument.</p>

<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_min()</a></code> and <code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_max()</a></code> select rows with highest or lowest values of a variable. Note that we first must choose only the values which are not NA.</li>
</ul>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
<a href="../reference/filter.html">filter</a>(!<a href="https://rdrr.io/r/base/NA.html">is.na</a>(<height)) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/slice.html">slice_max</a>(<height, n = <span class="fl">3)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 3 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Yarael…    264    <span style="color: #BB0000;">NA none       white      yellow            <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Tarfful    234   136 brown      brown      blue              <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Lama Su    229    88 none       grey       black             <span style="color: #BB0000;">NA male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

## <h3 id="select-columns-with-select">Select columns with <code>select()</code></h3>

<p>Often you work with large datasets with many columns but only a few are actually of interest to you. <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> allows you to rapidly zoom in on a useful subset using operations that usually only work on numeric variable positions:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><span class="co"># Select columns by name
<starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/select.html">select</a>(<hair_color, <skin_color, <eye_color)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 3
<span class="co">#&gt;   hair_color skin_color  eye_color
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;       <span style="color: #949494; font-style: italic;">&lt;chr&gt;    
<span class="co">#&gt; <span style="color: #BCBCBC;">1 blond      fair        blue     
<span class="co">#&gt; <span style="color: #BCBCBC;">2 <span style="color: #BB0000;">NA         gold        yellow   
<span class="co">#&gt; <span style="color: #BCBCBC;">3 <span style="color: #BB0000;">NA         white, blue red      
<span class="co">#&gt; <span style="color: #BCBCBC;">4 none       white       yellow   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co"># Select all columns between hair_color and eye_color (inclusive)
<starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/select.html">select</a>(<hair_color:<eye_color)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 3
<span class="co">#&gt;   hair_color skin_color  eye_color
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;       <span style="color: #949494; font-style: italic;">&lt;chr&gt;    
<span class="co">#&gt; <span style="color: #BCBCBC;">1 blond      fair        blue     
<span class="co">#&gt; <span style="color: #BCBCBC;">2 <span style="color: #BB0000;">NA         gold        yellow   
<span class="co">#&gt; <span style="color: #BCBCBC;">3 <span style="color: #BB0000;">NA         white, blue red      
<span class="co">#&gt; <span style="color: #BCBCBC;">4 none       white       yellow   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co"># Select all columns except those from hair_color to eye_color (inclusive)
<starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/select.html">select</a>(!(<hair_color:<eye_color))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 11
<span class="co">#&gt;   name     height  mass birth_year sex   gender homeworld species films
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;     <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt;      <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;  <span style="color: #949494; font-style: italic;">&lt;chr&gt;     <span style="color: #949494; font-style: italic;">&lt;chr&gt;   <span style="color: #949494; font-style: italic;">&lt;lis&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Sk…    172    77       19   male  mascu… Tatooine  Human   <span style="color: #949494;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO       167    75      112   none  mascu… Tatooine  Droid   <span style="color: #949494;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2        96    32       33   none  mascu… Naboo     Droid   <span style="color: #949494;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth V…    202   136       41.9 male  mascu… Tatooine  Human   <span style="color: #949494;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 2 more variables: vehicles &lt;list&gt;, starships &lt;list&gt;
<span class="co"># Select all columns ending with color
<starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/select.html">select</a>(<a href="https://tidyselect.r-lib.org/reference/starts_with.html">ends_with</a>("color"))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 3
<span class="co">#&gt;   hair_color skin_color  eye_color
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;       <span style="color: #949494; font-style: italic;">&lt;chr&gt;    
<span class="co">#&gt; <span style="color: #BCBCBC;">1 blond      fair        blue     
<span class="co">#&gt; <span style="color: #BCBCBC;">2 <span style="color: #BB0000;">NA         gold        yellow   
<span class="co">#&gt; <span style="color: #BCBCBC;">3 <span style="color: #BB0000;">NA         white, blue red      
<span class="co">#&gt; <span style="color: #BCBCBC;">4 none       white       yellow   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre></div>

<p>There are a number of helper functions you can use within <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>, like <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">starts_with()</a></code>, <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">ends_with()</a></code>, <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">matches()</a></code> and <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">contains()</a></code>. These let you quickly match larger blocks of variables that meet some criterion. See <code><a href="https://dplyr.tidyverse.org/reference/select.html">?select</a></code> for more details.</p>

<p>You can rename variables with <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> by using named arguments:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/select.html">select</a>(home_world = <homeworld)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 1
<span class="co">#&gt;   home_world
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;     
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Tatooine  
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Tatooine  
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Naboo     
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Tatooine  
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p>But because <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> drops all the variables not explicitly mentioned, it’s not that useful. Instead, use <code><a href="https://dplyr.tidyverse.org/reference/rename.html">rename()</a></code>:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/rename.html">rename</a>(home_world = <homeworld)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 14
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke S…    172    77 blond      fair       blue            19   male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow         112   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red             33   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth …    202   136 none       white      yellow          41.9 male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, home_world &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

## <h3 id="add-new-columns-with-mutate">Add new columns with <code>mutate()</code></h3>

<p>Besides selecting sets of existing columns, it’s often useful to add new columns that are functions of existing columns. This is the job of
<code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/mutate.html">mutate</a>(height_m = <height / <span class="fl">100)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 15
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke S…    172    77 blond      fair       blue            19   male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow         112   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red             33   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth …    202   136 none       white      yellow          41.9 male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 7 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;, height_m &lt;dbl&gt;</code></pre>

<p>We can’t see the height in meters we just calculated, but we can fix that using a select command.</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/mutate.html">mutate</a>(height_m = <height / <span class="fl">100) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/select.html">select</a>(<height_m, <height, <a href="https://tidyselect.r-lib.org/reference/everything.html">everything</a>())
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 15
<span class="co">#&gt;   height_m height name            mass hair_color skin_color  eye_color
<span class="co">#&gt;      <span style="color: #949494; font-style: italic;">&lt;dbl&gt;  <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;       <span style="color: #949494; font-style: italic;">&lt;chr&gt;    
<span class="co">#&gt; <span style="color: #BCBCBC;">1     1.72    172 Luke Skywalker    77 blond      fair        blue     
<span class="co">#&gt; <span style="color: #BCBCBC;">2     1.67    167 C-3PO             75 <span style="color: #BB0000;">NA         gold        yellow   
<span class="co">#&gt; <span style="color: #BCBCBC;">3     0.96     96 R2-D2             32 <span style="color: #BB0000;">NA         white, blue red      
<span class="co">#&gt; <span style="color: #BCBCBC;">4     2.02    202 Darth Vader      136 none       white       yellow   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 8 more variables: birth_year &lt;dbl&gt;, sex &lt;chr&gt;, gender &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   homeworld &lt;chr&gt;, species &lt;chr&gt;, films &lt;list&gt;, vehicles &lt;list&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   starships &lt;list&gt;</code></pre>

<p><code><a href="https://dplyr.tidyverse.org/reference/mutate.html">dplyr::mutate()</a></code> is similar to the base <code><a href="https://rdrr.io/r/base/transform.html">transform()</a></code>, but allows you to refer to columns that you’ve just created:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/mutate.html">mutate</a>(
    height_m = <height / <span class="fl">100,
    BMI = <mass / (<height_m^<span class="fl">2)
  ) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/select.html">select</a>(<BMI, <a href="https://tidyselect.r-lib.org/reference/everything.html">everything</a>())
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 16
<span class="co">#&gt;     BMI name    height  mass hair_color skin_color eye_color birth_year
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1  26.0 Luke S…    172    77 blond      fair       blue            19  
<span class="co">#&gt; <span style="color: #BCBCBC;">2  26.9 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow         112  
<span class="co">#&gt; <span style="color: #BCBCBC;">3  34.7 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red             33  
<span class="co">#&gt; <span style="color: #BCBCBC;">4  33.3 Darth …    202   136 none       white      yellow          41.9
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 8 more variables: sex &lt;chr&gt;, gender &lt;chr&gt;, homeworld &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   species &lt;chr&gt;, films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   height_m &lt;dbl&gt;</code></pre>

<p>If you only want to keep the new variables, use <code>.keep = "none"</code>:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/mutate.html">mutate</a>(
    height_m = <height / <span class="fl">100,
    BMI = <mass / (<height_m^<span class="fl">2),
    .keep = "none"
  )
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 2
<span class="co">#&gt;   height_m   BMI
<span class="co">#&gt;      <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1     1.72  26.0
<span class="co">#&gt; <span style="color: #BCBCBC;">2     1.67  26.9
<span class="co">#&gt; <span style="color: #BCBCBC;">3     0.96  34.7
<span class="co">#&gt; <span style="color: #BCBCBC;">4     2.02  33.3
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

## <h3 id="change-column-order-with-relocate">Change column order with <code>relocate()</code></h3>

<p>Use a similar syntax as <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> to move blocks of columns at once</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/relocate.html">relocate</a>(<sex:<homeworld, .before = <height)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 14
<span class="co">#&gt;   name        sex   gender homeworld height  mass hair_color skin_color
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;       <span style="color: #949494; font-style: italic;">&lt;chr&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;  <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;     
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywa… male  mascu… Tatooine     172    77 blond      fair      
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO       none  mascu… Tatooine     167    75 <span style="color: #BB0000;">NA         gold      
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       none  mascu… Naboo         96    32 <span style="color: #BB0000;">NA         white, bl…
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader male  mascu… Tatooine     202   136 none       white     
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: eye_color &lt;chr&gt;, birth_year &lt;dbl&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   species &lt;chr&gt;, films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;</code></pre>

## <h3 id="summarise-values-with-summarise">Summarise values with <code>summarise()</code></h3>

<p>The last verb is <code><a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise()</a></code>. It collapses a data frame to a single row.</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/summarise.html">summarise</a>(height = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<height, na.rm = <span class="cn">TRUE))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 1 × 1
<span class="co">#&gt;   height
<span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1   175.</code></pre>

<p>It’s not that useful until we learn the <code><a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by()</a></code> verb below.</p>

## <h3 id="commonalities">Commonalities</h3>

<p>You may have noticed that the syntax and function of all these verbs are very similar:</p>

<ul>
<li><p>The first argument is a data frame.</p></li>
<li><p>The subsequent arguments describe what to do with the data frame. You can refer to columns in the data frame directly without using <code>$</code>.</p></li>
<li><p>The result is a new data frame</p></li>
</ul>

<p>Together these properties make it easy to chain together multiple simple steps to achieve a complex result.</p>

<p>These five functions provide the basis of a language of data manipulation. At the most basic level, you can only alter a tidy data frame in five useful ways: you can reorder the rows (<code><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange()</a></code>), pick observations and variables of interest (<code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> and <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>), add new variables that are functions of existing variables (<code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>), or collapse many values to a summary (<code><a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise()</a></code>).</p>

## <h2 id="combining-functions-with">Combining functions with <code>%&gt;%</code></h2>

<p>The dplyr API is functional in the sense that function calls don’t have side-effects. You must always save their results. This doesn’t lead
to particularly elegant code, especially if you want to do many operations at once. You either have to do it step-by-step:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><a1 &lt;- <a href="../reference/group_by.html">group_by</a>(<starwars, <species, <sex)
<a2 &lt;- <a href="../reference/select.html">select</a>(<a1, <height, <mass)
<a3 &lt;- <a href="../reference/summarise.html">summarise</a>(<a2,
  height = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<height, na.rm = <span class="cn">TRUE),
  mass = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<mass, na.rm = <span class="cn">TRUE)
)</code></pre>

<p>Or if you don’t want to name the intermediate results, you need to wrap the function calls inside each other:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><a href="../reference/summarise.html">summarise</a>(
  <a href="../reference/select.html">select</a>(
    <a href="../reference/group_by.html">group_by</a>(<starwars, <species, <sex),
    <height, <mass
  ),
  height = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<height, na.rm = <span class="cn">TRUE),
  mass = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<mass, na.rm = <span class="cn">TRUE)
)
<span class="co">#&gt; Adding missing grouping variables: `species`, `sex`
<span class="co">#&gt; `summarise()` has grouped output by 'species'. You can override using
<span class="co">#&gt; the `.groups` argument.
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 41 × 4
<span class="co">#&gt; <span style="color: #949494;"># Groups:   species [38]
<span class="co">#&gt;   species  sex   height  mass
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;  <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Aleena   male      79    15
<span class="co">#&gt; <span style="color: #BCBCBC;">2 Besalisk male     198   102
<span class="co">#&gt; <span style="color: #BCBCBC;">3 Cerean   male     198    82
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Chagrian male     196   <span style="color: #BB0000;">NaN
<span class="co">#&gt; <span style="color: #949494;"># ℹ 37 more rows</code></pre>

<p>This is difficult to read because the order of the operations is from inside to out. Thus, the arguments are a long way away from the
function. To get around this problem, dplyr provides the <code>%&gt;%</code> operator from magrittr. <code>x %&gt;% f(y)</code> turns into <code>f(x, y)</code> so you can use it to rewrite multiple operations that you can read left-to-right, top-to-bottom (reading the pipe operator as “then”):</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/group_by.html">group_by</a>(<species, <sex) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/select.html">select</a>(<height, <mass) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="../reference/summarise.html">summarise</a>(
    height = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<height, na.rm = <span class="cn">TRUE),
    mass = <a href="https://rdrr.io/r/base/mean.html">mean</a>(<mass, na.rm = <span class="cn">TRUE)
  )</code></pre>

# <h2 id="patterns-of-operations">Patterns of operations</h2>

<p>The dplyr verbs can be classified by the type of operations they accomplish (we sometimes speak of their <strong>semantics</strong>, i.e., their meaning). It’s helpful to have a good grasp of the difference between select and mutate operations.</p>

## <h3 id="selecting-operations">Selecting operations</h3>

<p>One of the appealing features of dplyr is that you can refer to columns from the tibble as if they were regular variables. However, the syntactic uniformity of referring to bare column names hides semantical differences across the verbs. A column symbol supplied to <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> does not have the same meaning as the same symbol supplied to <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>.</p>

<p>Selecting operations expect column names and positions. Hence, when you call <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> with bare variable names, they actually represent their own positions in the tibble. The following calls are completely equivalent from dplyr’s point of view:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><span class="co"># `name` represents the integer 1
<a href="../reference/select.html">select</a>(<starwars, <name)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 1
<span class="co">#&gt;   name          
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;         
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO         
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2         
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<a href="../reference/select.html">select</a>(<starwars, <span class="fl">1)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 1
<span class="co">#&gt;   name          
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;         
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO         
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2         
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p>By the same token, this means that you cannot refer to variables from the surrounding context if they have the same name as one of the columns. In the following example, <code>height</code> still represents 2, not 5:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><height &lt;- <span class="fl">5
<a href="../reference/select.html">select</a>(<starwars, <height)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 1
<span class="co">#&gt;   height
<span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1    172
<span class="co">#&gt; <span style="color: #BCBCBC;">2    167
<span class="co">#&gt; <span style="color: #BCBCBC;">3     96
<span class="co">#&gt; <span style="color: #BCBCBC;">4    202
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p>One useful subtlety is that this only applies to bare names and to selecting calls like <code>c(height, mass)</code> or <code>height:mass</code>. In all other cases, the columns of the data frame are not put in scope. This allows you to refer to contextual variables in selection helpers:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><name &lt;- "color"
<a href="../reference/select.html">select</a>(<starwars, <a href="https://tidyselect.r-lib.org/reference/starts_with.html">ends_with</a>(<name))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 3
<span class="co">#&gt;   hair_color skin_color  eye_color
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;       <span style="color: #949494; font-style: italic;">&lt;chr&gt;    
<span class="co">#&gt; <span style="color: #BCBCBC;">1 blond      fair        blue     
<span class="co">#&gt; <span style="color: #BCBCBC;">2 <span style="color: #BB0000;">NA         gold        yellow   
<span class="co">#&gt; <span style="color: #BCBCBC;">3 <span style="color: #BB0000;">NA         white, blue red      
<span class="co">#&gt; <span style="color: #BCBCBC;">4 none       white       yellow   
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p>These semantics are usually intuitive. But note the subtle difference:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><name &lt;- <span class="fl">5
<a href="https://dplyr.tidyverse.org/reference/select.html">select</a>(<starwars, <name, <a href="https://rdrr.io/r/base/identity.html">identity</a>(<name))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 2
<span class="co">#&gt;   name           skin_color 
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;chr&gt;      
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker fair       
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO          gold       
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2          white, blue
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader    white      
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p>In the first argument, <code>name</code> represents its own position <code>1</code>. In the second argument, <code>name</code> is evaluated in the surrounding context and represents the fifth column.</p>

<p>For a long time, <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> used to only understand column positions. Counting from dplyr 0.6, it now understands column names as well. This makes it a bit easier to program with <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><vars &lt;- <a href="https://rdrr.io/r/base/c.html">c</a>("name", "height")
<a href="../reference/select.html">select</a>(<starwars, <a href="https://tidyselect.r-lib.org/reference/all_of.html">all_of</a>(<vars), "mass")
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 3
<span class="co">#&gt;   name           height  mass
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;           <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker    172    77
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO             167    75
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2              96    32
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader       202   136
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

## <h3 id="mutating-operations">Mutating operations</h3>

<p>Mutate semantics are quite different from selection semantics.
Whereas <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> expects column names or positions, <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> expects <em>column vectors</em>. We will set up a smaller tibble to use for our examples.</p>

<pre class="downlit sourceCode r"><code class="sourceCode R">df &lt;- starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a> <a href="../reference/select.html">select</a>(name, height, mass)</code></pre>

<p>When we use <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>, the bare column names stand for their own positions in the tibble. For <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> on the other hand, column symbols represent the actual column vectors stored in the tibble. Consider what happens if we give a string or a number to <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><a href="../reference/mutate.html">mutate</a>(<df, "height", <span class="fl">2)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 5
<span class="co">#&gt;   name           height  mass `"height"`   `2`
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;           <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker    172    77 height         2
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO             167    75 height         2
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2              96    32 height         2
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader       202   136 height         2
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p><code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> gets length-1 vectors that it interprets as new columns in the data frame. These vectors are recycled so they match the number of rows. That’s why it doesn’t make sense to supply expressions like <code>"height" + 10</code> to <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>. This amounts to adding 10 to a string! The correct expression is:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><a href="../reference/mutate.html">mutate</a>(<df, <height + <span class="fl">10)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 4
<span class="co">#&gt;   name           height  mass `height + 10`
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;           <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt;         <span style="color: #949494; font-style: italic;">&lt;dbl&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker    172    77           182
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO             167    75           177
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2              96    32           106
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader       202   136           212
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>

<p>In the same way, you can unquote values from the context if these values represent a valid column. They must be either length 1 (they then get recycled) or have the same length as the number of rows. In the following example we create a new vector that we add to the data frame:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><var &lt;- <a href="https://rdrr.io/r/base/seq.html">seq</a>(<span class="fl">1, <a href="https://rdrr.io/r/base/nrow.html">nrow</a>(<df))
<a href="../reference/mutate.html">mutate</a>(<df, new = <var)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 4
<span class="co">#&gt;   name           height  mass   new
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;           <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;int&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker    172    77     1
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO             167    75     2
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2              96    32     3
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader       202   136     4
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre></div>

<p>A case in point is <code><a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by()</a></code>. While you might think it has select semantics, it actually has mutate semantics. This is quite handy as it allows to group by a modified column:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><a href="../reference/group_by.html">group_by</a>(<starwars, <sex)
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 14
<span class="co">#&gt; <span style="color: #949494;"># Groups:   sex [5]
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke S…    172    77 blond      fair       blue            19   male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow         112   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red             33   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth …    202   136 none       white      yellow          41.9 male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;
<a href="../reference/group_by.html">group_by</a>(<starwars, sex = <a href="https://rdrr.io/r/base/factor.html">as.factor</a>(<sex))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 14
<span class="co">#&gt; <span style="color: #949494;"># Groups:   sex [5]
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;fct&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke S…    172    77 blond      fair       blue            19   male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow         112   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red             33   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth …    202   136 none       white      yellow          41.9 male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 6 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;
<a href="../reference/group_by.html">group_by</a>(<starwars, height_binned = <a href="https://rdrr.io/r/base/cut.html">cut</a>(<height, <span class="fl">3))
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 15
<span class="co">#&gt; <span style="color: #949494;"># Groups:   height_binned [4]
<span class="co">#&gt;   name    height  mass hair_color skin_color eye_color birth_year sex  
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;    <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;      <span style="color: #949494; font-style: italic;">&lt;chr&gt;          <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke S…    172    77 blond      fair       blue            19   male 
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO      167    75 <span style="color: #BB0000;">NA         gold       yellow         112   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2       96    32 <span style="color: #BB0000;">NA         white, bl… red             33   none 
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth …    202   136 none       white      yellow          41.9 male 
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows
<span class="co">#&gt; <span style="color: #949494;"># ℹ 7 more variables: gender &lt;chr&gt;, homeworld &lt;chr&gt;, species &lt;chr&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   films &lt;list&gt;, vehicles &lt;list&gt;, starships &lt;list&gt;,
<span class="co">#&gt; <span style="color: #949494;">#   height_binned &lt;fct&gt;</code></pre>

<p>This is why you can’t supply a column name to <code><a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by()</a></code>. This amounts to creating a new column containing the string recycled to the number of rows:</p>

<pre class="downlit sourceCode r"><code class="sourceCode R"><a href="../reference/group_by.html">group_by</a>(<df, "month")
<span class="co">#&gt; <span style="color: #949494;"># A tibble: 87 × 4
<span class="co">#&gt; <span style="color: #949494;"># Groups:   "month" [1]
<span class="co">#&gt;   name           height  mass `"month"`
<span class="co">#&gt;   <span style="color: #949494; font-style: italic;">&lt;chr&gt;           <span style="color: #949494; font-style: italic;">&lt;int&gt; <span style="color: #949494; font-style: italic;">&lt;dbl&gt; <span style="color: #949494; font-style: italic;">&lt;chr&gt;    
<span class="co">#&gt; <span style="color: #BCBCBC;">1 Luke Skywalker    172    77 month    
<span class="co">#&gt; <span style="color: #BCBCBC;">2 C-3PO             167    75 month    
<span class="co">#&gt; <span style="color: #BCBCBC;">3 R2-D2              96    32 month    
<span class="co">#&gt; <span style="color: #BCBCBC;">4 Darth Vader       202   136 month    
<span class="co">#&gt; <span style="color: #949494;"># ℹ 83 more rows</code></pre>